# HW3 — Two retrievers and a model: what RAG actually sends

**Serbina Sofiya**
**S23070210**
**CSS4007-ENG-10**
**https://github.com/vesta-12/hw3-vesta-12**

## AI tool disclosure

### AI Assistance Disclosure

I used ChatGPT as a study coding assistant while completing this assignment.

It helped me:
- check manual BM25 and cosine-similarity calculations;
- structure the retrieval and RAG code;
- review the outputs and help formulate explanations based on my own results.

All code was run by me, and all reported retrieval results, scores, token counts and model outputs in this notebook come from my own executed notebook.

>

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.
The instructions for every section are in `README.md`.

In [2]:
!git clone https://github.com/vesta-12/hw3-vesta-12.git
%cd hw3-vesta-12
!pip install -r requirements.txt

Cloning into 'hw3-vesta-12'...
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 18 (delta 5), reused 16 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (18/18), 45.52 KiB | 991.00 KiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/hw3-vesta-12
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 30.0 MB/s eta 0:00:00


In [3]:
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
openai_api_key = userdata.get("OPENAI_API_KEY")

print("HF token found:", bool(hf_token))
print("OpenAI key found:", bool(openai_api_key))

HF token found: True
OpenAI key found: True


In [4]:
%cd /content/hw3-vesta-12
!pwd

/content/hw3-vesta-12
/content/hw3-vesta-12


---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

### A1 — BM25

#### Tokenisation

Following the required rules, I lowercase the text, split on spaces and hyphens,
remove punctuation, and keep stop words.

D1:
large, language, models, can, generate, text, and, answer, questions

|D1| = 9

D2:
retrieval, augmented, generation, retrieves, relevant, documents, before, generating, an, answer

|D2| = 10

D3:
embeddings, represent, text, as, numerical, vectors

|D3| = 6

Query:
how, does, rag, retrieve, information

#### Document lengths

N = 3

|D1| = 9
|D2| = 10
|D3| = 6

avgdl = (9 + 10 + 6) / 3
      = 25 / 3
      = 8.3333

### A1(a) — BM25 without stemming

#### Term frequency

The query terms are:

how, does, rag, retrieve, information

I count only exact token matches.

| Query term | D1 | D2 | D3 |
|---|---:|---:|---:|
| how | 0 | 0 | 0 |
| does | 0 | 0 | 0 |
| rag | 0 | 0 | 0 |
| retrieve | 0 | 0 | 0 |
| information | 0 | 0 | 0 |

D2 contains the words `retrieval` and `retrieves`, but neither is exactly equal to the query token `retrieve`.

Therefore, every query term has frequency 0 in every document.

#### Document frequency and IDF

Document frequency `n(t)` is the number of documents that contain the query term.

Since none of the query terms appears exactly in D1, D2, or D3:

| Query term | n(t) |
|---|---:|
| how | 0 |
| does | 0 |
| rag | 0 |
| retrieve | 0 |
| information | 0 |

The IDF formula is:

\[
idf(t) = \ln\left(\frac{N-n(t)+0.5}{n(t)+0.5}\right)
\]

Here, \(N=3\).

For every query term, \(n(t)=0\), so:

\[
idf(t)=\ln\left(\frac{3-0+0.5}{0+0.5}\right)
\]

\[
=\ln\left(\frac{3.5}{0.5}\right)
\]

\[
=\ln(7)
\]

\[
\approx 1.9459
\]

Therefore:

| Query term | n(t) | IDF |
|---|---:|---:|
| how | 0 | 1.9459 |
| does | 0 | 1.9459 |
| rag | 0 | 1.9459 |
| retrieve | 0 | 1.9459 |
| information | 0 | 1.9459 |

#### BM25 scores

The BM25 formula is:

\[
score(q,d)=\sum_{t \in q}
idf(t)\cdot
\frac{f(t,d)(k_1+1)}
{f(t,d)+k_1\left(1-b+b\frac{|d|}{avgdl}\right)}
\]

where:

\[
k_1=1.5,\qquad b=0.75,\qquad avgdl=8.3333
\]

For all query terms in all three documents:

\[
f(t,d)=0
\]

Therefore, the numerator of every term contribution is:

\[
f(t,d)(k_1+1)=0(1.5+1)=0
\]

So every query term contributes 0 to the BM25 score.

For D1:

\[
score(q,D1)=0
\]

For D2:

\[
score(q,D2)=0
\]

For D3:

\[
score(q,D3)=0
\]

| Document | Length | BM25 score |
|---|---:|---:|
| D1 | 9 | 0 |
| D2 | 10 | 0 |
| D3 | 6 | 0 |

#### Ranking

All three documents have the same score:

\[
D1 = D2 = D3
\]

There is no unique first-ranked document.

#### Explanation

At first this result looks surprising because D2 is clearly related to the query.

However, BM25 without stemming uses exact token matches.

The query contains:

`retrieve`

but D2 contains:

`retrieval` and `retrieves`

These are different tokens under the required tokenisation rules.

Therefore, none of the query terms occurs exactly in any document, so all three BM25 scores are 0.

This is why BM25 cannot choose a unique first document in part (a).

### A1(b) — BM25 with stemming

The following stems are applied wherever these words appear:

- retrieve, retrieves, retrieval → `retriev`
- generate, generating, generation → `gener`
- information → `inform`
- does → `doe`

All other words stay unchanged.

#### Stemmed query

Original query:

`how, does, rag, retrieve, information`

After stemming:

`how, doe, rag, retriev, inform`

#### Stemmed documents

D1:

`large, language, models, can, gener, text, and, answer, questions`

D2:

`retriev, augmented, gener, retriev, relevant, documents, before, gener, an, answer`

D3:

`embeddings, represent, text, as, numerical, vectors`

The document lengths do not change:

\[
|D1|=9,\qquad |D2|=10,\qquad |D3|=6
\]

and:

\[
avgdl=8.3333
\]

#### Term frequency after stemming

| Query term | D1 | D2 | D3 |
|---|---:|---:|---:|
| how | 0 | 0 | 0 |
| doe | 0 | 0 | 0 |
| rag | 0 | 0 | 0 |
| retriev | 0 | 2 | 0 |
| inform | 0 | 0 | 0 |

The stem `retriev` occurs twice in D2 because both `retrieval` and `retrieves` are converted to the same stem.

#### Document frequency and IDF after stemming

The stemmed query terms are:

`how, doe, rag, retriev, inform`

Their document frequencies are:

| Query term | n(t) |
|---|---:|
| how | 0 |
| doe | 0 |
| rag | 0 |
| retriev | 1 |
| inform | 0 |

Only `retriev` appears in a document, and it appears in D2.

The IDF formula is:

\[
idf(t)=\ln\left(\frac{N-n(t)+0.5}{n(t)+0.5}\right)
\]

where:

\[
N=3
\]

For terms with \(n(t)=0\):

\[
idf(t)=\ln\left(\frac{3-0+0.5}{0+0.5}\right)
\]

\[
=\ln(7)
\]

\[
\approx 1.9459
\]

For `retriev`, \(n(t)=1\):

\[
idf(retriev)=\ln\left(\frac{3-1+0.5}{1+0.5}\right)
\]

\[
=\ln\left(\frac{2.5}{1.5}\right)
\]

\[
=\ln(1.6667)
\]

\[
\approx 0.5108
\]

Therefore:

| Query term | n(t) | IDF |
|---|---:|---:|
| how | 0 | 1.9459 |
| doe | 0 | 1.9459 |
| rag | 0 | 1.9459 |
| retriev | 1 | 0.5108 |
| inform | 0 | 1.9459 |

#### BM25 scores after stemming

The BM25 formula is:

\[
score(q,d)=\sum_{t \in q}
idf(t)\cdot
\frac{f(t,d)(k_1+1)}
{f(t,d)+k_1\left(1-b+b\frac{|d|}{avgdl}\right)}
\]

where:

\[
k_1=1.5,\qquad b=0.75,\qquad avgdl=8.3333
\]

For D1 and D3, none of the stemmed query terms occurs, so:

\[
score(q,D1)=0
\]

\[
score(q,D3)=0
\]

For D2, only the stem `retriev` contributes.

For `retriev`:

\[
f(retriev,D2)=2
\]

\[
idf(retriev)=0.5108
\]

First calculate the document length normalization:

\[
1-b+b\frac{|D2|}{avgdl}
\]

\[
=1-0.75+0.75\frac{10}{8.3333}
\]

\[
=0.25+0.75(1.2)
\]

\[
=0.25+0.9
\]

\[
=1.15
\]

Then:

\[
k_1\left(1-b+b\frac{|D2|}{avgdl}\right)
\]

\[
=1.5(1.15)
\]

\[
=1.725
\]

The denominator is:

\[
f(t,d)+1.725
\]

\[
=2+1.725
\]

\[
=3.725
\]

The numerator is:

\[
f(t,d)(k_1+1)
\]

\[
=2(1.5+1)
\]

\[
=2(2.5)
\]

\[
=5
\]

Therefore:

\[
score(q,D2)
=
0.5108\cdot\frac{5}{3.725}
\]

\[
\approx 0.6857
\]

So the BM25 scores are:

| Document | BM25 score |
|---|---:|
| D1 | 0 |
| D2 | 0.6857 |
| D3 | 0 |

#### Ranking

\[
D2 > D1 = D3
\]

Therefore, with stemming, BM25 retrieves D2 first.

#### A1 conclusion

Without stemming, all three documents receive a BM25 score of 0 because none of the query terms matches any document token exactly.

In particular, the query contains `retrieve`, while D2 contains `retrieval` and `retrieves`.

After stemming, these three word forms become the same stem, `retriev`.

As a result, D2 receives a positive BM25 score and is ranked first.

So:

- without stemming: D1 = D2 = D3
- with stemming: D2 > D1 = D3

### A2 — Cosine similarity

The vectors are:

\[
Q=[0.8,0.2,0.5]
\]

\[
D1=[0.7,0.1,0.4]
\]

\[
D2=[0.9,0.3,0.6]
\]

\[
D3=[0.1,0.8,0.2]
\]

Cosine similarity is calculated as:

\[
\cos(Q,D)=\frac{Q\cdot D}{||Q||\cdot||D||}
\]

#### Magnitude of the query

\[
||Q||=\sqrt{0.8^2+0.2^2+0.5^2}
\]

\[
=\sqrt{0.64+0.04+0.25}
\]

\[
=\sqrt{0.93}
\]

\[
\approx 0.9644
\]

#### D1

Dot product:

\[
Q\cdot D1=(0.8)(0.7)+(0.2)(0.1)+(0.5)(0.4)
\]

\[
=0.56+0.02+0.20
\]

\[
=0.78
\]

Magnitude of D1:

\[
||D1||=\sqrt{0.7^2+0.1^2+0.4^2}
\]

\[
=\sqrt{0.49+0.01+0.16}
\]

\[
=\sqrt{0.66}
\]

\[
\approx 0.8124
\]

Cosine similarity:

\[
\cos(Q,D1)=\frac{0.78}{(0.9644)(0.8124)}
\]

\[
\approx 0.9956
\]

#### D2

Dot product:

\[
Q\cdot D2=(0.8)(0.9)+(0.2)(0.3)+(0.5)(0.6)
\]

\[
=0.72+0.06+0.30
\]

\[
=1.08
\]

Magnitude of D2:

\[
||D2||=\sqrt{0.9^2+0.3^2+0.6^2}
\]

\[
=\sqrt{0.81+0.09+0.36}
\]

\[
=\sqrt{1.26}
\]

\[
\approx 1.1225
\]

Cosine similarity:

\[
\cos(Q,D2)=\frac{1.08}{(0.9644)(1.1225)}
\]

\[
\approx 0.9977
\]

#### D3

Dot product:

\[
Q\cdot D3=(0.8)(0.1)+(0.2)(0.8)+(0.5)(0.2)
\]

\[
=0.08+0.16+0.10
\]

\[
=0.34
\]

Magnitude of D3:

\[
||D3||=\sqrt{0.1^2+0.8^2+0.2^2}
\]

\[
=\sqrt{0.01+0.64+0.04}
\]

\[
=\sqrt{0.69}
\]

\[
\approx 0.8307
\]

Cosine similarity:

\[
\cos(Q,D3)=\frac{0.34}{(0.9644)(0.8307)}
\]

\[
\approx 0.4244
\]

#### A2 summary

| Document | Dot product | Magnitude of query | Magnitude of document | Cosine similarity |
|---|---:|---:|---:|---:|
| D1 | 0.78 | 0.9644 | 0.8124 | 0.9956 |
| D2 | 1.08 | 0.9644 | 1.1225 | 0.9977 |
| D3 | 0.34 | 0.9644 | 0.8307 | 0.4244 |

#### Ranking

\[
D2 > D1 > D3
\]

Therefore, D2 is the document most similar to the query.

#### Comparison with BM25

The cosine similarity ranking is:

\[
D2 > D1 > D3
\]

The BM25 ranking without stemming was:

\[
D1 = D2 = D3
\]

The BM25 ranking with stemming was:

\[
D2 > D1 = D3
\]

So cosine similarity agrees with stemmed BM25 that D2 should be ranked first, but cosine similarity also distinguishes D1 from D3.

#### Why BM25 and embedding retrieval disagree

BM25 relies on matching query terms with document terms.

In this example, the query contains the word `retrieve`, but D2 contains `retrieval` and `retrieves`. Without stemming, BM25 treats these as different tokens, so it gives D2 no lexical match.

The embedding vectors can still represent the semantic similarity between the query and D2. D2 describes Retrieval-Augmented Generation and retrieving relevant documents, so its vector is very close to the query vector even though the exact word `retrieve` is not present.

This is why D2 is ranked first by cosine similarity, while BM25 without stemming cannot distinguish between the three documents.

In [5]:
import math

# ---------- A1: BM25 check ----------

k1 = 1.5
b = 0.75

doc_lengths = {
    "D1": 9,
    "D2": 10,
    "D3": 6
}

avgdl = sum(doc_lengths.values()) / len(doc_lengths)

print("A1 BM25 CHECK")
print("avgdl =", avgdl)

# A1(a): without stemming
scores_without_stemming = {
    "D1": 0.0,
    "D2": 0.0,
    "D3": 0.0
}

print("\nWithout stemming:")
for doc, score in scores_without_stemming.items():
    print(doc, "=", score)

# A1(b): with stemming
N = 3
n_retriev = 1
f_retriev_D2 = 2

idf_retriev = math.log(
    (N - n_retriev + 0.5) /
    (n_retriev + 0.5)
)

d2_length = doc_lengths["D2"]

d2_score = (
    idf_retriev
    * f_retriev_D2
    * (k1 + 1)
    /
    (
        f_retriev_D2
        + k1 * (
            1 - b
            + b * d2_length / avgdl
        )
    )
)

print("\nWith stemming:")
print("IDF(retriev) =", round(idf_retriev, 4))
print("D1 =", 0.0)
print("D2 =", round(d2_score, 4))
print("D3 =", 0.0)


# ---------- A2: cosine similarity check ----------

query = [0.8, 0.2, 0.5]

documents = {
    "D1": [0.7, 0.1, 0.4],
    "D2": [0.9, 0.3, 0.6],
    "D3": [0.1, 0.8, 0.2]
}

def magnitude(vector):
    return math.sqrt(sum(x ** 2 for x in vector))

def dot_product(a, b):
    return sum(x * y for x, y in zip(a, b))

query_magnitude = magnitude(query)

print("\nA2 COSINE SIMILARITY CHECK")
print("Query magnitude =", round(query_magnitude, 4))

results = []

for name, vector in documents.items():
    dot = dot_product(query, vector)
    doc_magnitude = magnitude(vector)
    cosine = dot / (query_magnitude * doc_magnitude)

    results.append((name, cosine))

    print(f"\n{name}")
    print("dot product =", round(dot, 4))
    print("document magnitude =", round(doc_magnitude, 4))
    print("cosine similarity =", round(cosine, 4))

print("\nRanking:")
for rank, (name, cosine) in enumerate(
    sorted(results, key=lambda x: x[1], reverse=True),
    start=1
):
    print(rank, name, round(cosine, 4))

A1 BM25 CHECK
avgdl = 8.333333333333334

Without stemming:
D1 = 0.0
D2 = 0.0
D3 = 0.0

With stemming:
IDF(retriev) = 0.5108
D1 = 0.0
D2 = 0.6857
D3 = 0.0

A2 COSINE SIMILARITY CHECK
Query magnitude = 0.9644

D1
dot product = 0.78
document magnitude = 0.8124
cosine similarity = 0.9956

D2
dot product = 1.08
document magnitude = 1.1225
cosine similarity = 0.9977

D3
dot product = 0.34
document magnitude = 0.8307
cosine similarity = 0.4244

Ranking:
1 D2 0.9977
2 D1 0.9956
3 D3 0.4244


## Part B — Library retrievers

### B1 — BM25 with rank_bm25

For BM25 tokenisation I use the following rules:

- lowercase all text;
- extract words with a regular expression;
- keep hyphenated tokens such as `GR-305` together;
- keep stop words;
- do not apply stemming.

I keep hyphenated form codes together because codes such as `GR-305`, `GR-104`,
and `GR-210` can be important exact lexical matches in this corpus.

In [6]:
import json
import re

from rank_bm25 import BM25Okapi


# Load corpus
corpus = []

with open("data/corpus.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        corpus.append(json.loads(line))


# Load fixed queries
with open("data/queries.json", "r", encoding="utf-8") as f:
    query_data = json.load(f)

fixed_queries = query_data["queries"]


print("Number of chunks:", len(corpus))
print("Number of fixed queries:", len(fixed_queries))

print("\nFirst chunk:")
print("ID:", corpus[0]["id"])
print("Title:", corpus[0]["title"])
print("Language:", corpus[0]["lang"])
print("Text:", corpus[0]["text"][:200] + "...")

Number of chunks: 74
Number of fixed queries: 8

First chunk:
ID: KZ-01
Title: Kazakhstan (part 1)
Language: en
Text: Kazakhstan, officially the Republic of Kazakhstan, is a landlocked country situated primarily in Central Asia, with a portion of its territory extending into Eastern Europe. It borders Russia to the n...


In [7]:
def tokenize(text):
    return re.findall(r"\b[\w]+(?:-[\w]+)*\b", text.lower())


# Small check of the tokenizer
examples = [
    "Form GR-305 is used for appeals.",
    "Retrieval-Augmented Generation",
    "Өтінішті қашанға дейін тапсыру керек?"
]

for example in examples:
    print(example)
    print(tokenize(example))
    print()

Form GR-305 is used for appeals.
['form', 'gr-305', 'is', 'used', 'for', 'appeals']

Retrieval-Augmented Generation
['retrieval-augmented', 'generation']

Өтінішті қашанға дейін тапсыру керек?
['өтінішті', 'қашанға', 'дейін', 'тапсыру', 'керек']



In [8]:
tokenized_corpus = [
    tokenize(chunk["text"])
    for chunk in corpus
]

bm25 = BM25Okapi(
    tokenized_corpus,
    k1=1.5,
    b=0.75
)

print("BM25 index built successfully.")
print("Indexed chunks:", len(tokenized_corpus))

BM25 index built successfully.
Indexed chunks: 74


In [9]:
def first_line(text):
    for line in text.splitlines():
        line = line.strip()
        if line:
            return line
    return ""


def bm25_top_k(query, k=3):
    query_tokens = tokenize(query)
    scores = bm25.get_scores(query_tokens)

    ranked_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )[:k]

    results = []

    for rank, i in enumerate(ranked_indices, start=1):
        results.append({
            "rank": rank,
            "id": corpus[i]["id"],
            "title": corpus[i]["title"],
            "first_line": first_line(corpus[i]["text"]),
            "score": float(scores[i])
        })

    return results

In [10]:
test_query = fixed_queries[0]

print("Query:", test_query["id"])
print(test_query["text"])
print("Relevant:", test_query["relevant"])

results = bm25_top_k(test_query["text"], k=3)

for result in results:
    print(
        f'{result["rank"]}. '
        f'{result["id"]} | '
        f'{result["first_line"]} | '
        f'score = {result["score"]:.4f}'
    )

Query: Q-01
What is form GR-305 used for?
Relevant: ['GO-08', 'GO-12', 'GO-23']
1. GO-08 | Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision they believe is wrong and which rule in this handbook supports their case. | score = 13.3065
2. GO-12 | A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using form GR-305. The appeal is read by a panel of three staff members who did not take part in the original decision, and the panel answers within 20 working days. An appeal reviews the decision on the evidence that was on file; new income information cannot be added through an appeal and needs a new application. | score = 6.4533
3. GO-06 | Form GR-104 is the application form for the need-based study grant. It is the only way to apply; an email

In [11]:
bm25_results = {}

for query in fixed_queries:
    query_id = query["id"]
    query_text = query["text"]
    relevant = query["relevant"]

    results = bm25_top_k(query_text, k=3)
    bm25_results[query_id] = results

    retrieved_ids = [result["id"] for result in results]
    hit_at_3 = any(chunk_id in relevant for chunk_id in retrieved_ids)

    print("=" * 100)
    print(f'{query_id}: {query_text}')
    print(f'Relevant chunks: {relevant}')
    print()

    for result in results:
        print(
            f'{result["rank"]}. '
            f'{result["id"]} | '
            f'{result["first_line"]} | '
            f'score = {result["score"]:.4f}'
        )

    print()
    print("hit@3:", hit_at_3)
    print()

Q-01: What is form GR-305 used for?
Relevant chunks: ['GO-08', 'GO-12', 'GO-23']

1. GO-08 | Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision they believe is wrong and which rule in this handbook supports their case. | score = 13.3065
2. GO-12 | A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using form GR-305. The appeal is read by a panel of three staff members who did not take part in the original decision, and the panel answers within 20 working days. An appeal reviews the decision on the evidence that was on file; new income information cannot be added through an appeal and needs a new application. | score = 6.4533
3. GO-06 | Form GR-104 is the application form for the need-based study grant. It is the only way to apply; an ema

In [12]:
bm25_hits = {}

for query in fixed_queries:
    query_id = query["id"]
    relevant = query["relevant"]

    retrieved_ids = [
        result["id"]
        for result in bm25_results[query_id]
    ]

    bm25_hits[query_id] = any(
        chunk_id in relevant
        for chunk_id in retrieved_ids
    )


print("BM25 hit@3 summary")
print("-" * 30)

for query_id, hit in bm25_hits.items():
    print(f"{query_id}: {hit}")

total_hits = sum(bm25_hits.values())

print()
print(f"Total: {total_hits} / {len(fixed_queries)}")

BM25 hit@3 summary
------------------------------
Q-01: True
Q-02: False
Q-03: True
Q-04: False
Q-05: True
Q-06: True
Q-07: False
Q-08: True

Total: 5 / 8


### B2 — EmbeddingGemma

For semantic retrieval I use `google/embeddinggemma-300m`
with SentenceTransformers.

The corpus chunks are encoded once with `encode_document()`,
while queries are encoded with `encode_query()`.

This distinction is important because EmbeddingGemma uses different
instruction prefixes for documents and queries.

Cosine similarity is then used to rank all corpus chunks for each query.

In [13]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "google/embeddinggemma-300m",
    token=hf_token
)

print("EmbeddingGemma loaded successfully.")

modules.json:   0%|          | 0.00/573 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/18.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.49k [00:00<?, ?B/s]

...

2_Dense/model.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

3_Dense/model.safetensors: reconstructing file:   0%|          |  0.00B / 9.44MB            

3_Dense/model.safetensors: downloading bytes:           |  0.00B            

EmbeddingGemma loaded successfully.


In [14]:
import json
import re

from rank_bm25 import BM25Okapi

# Load corpus
corpus = []

with open("data/corpus.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        corpus.append(json.loads(line))

# Load fixed queries
with open("data/queries.json", "r", encoding="utf-8") as f:
    query_data = json.load(f)

fixed_queries = query_data["queries"]

print("Number of chunks:", len(corpus))
print("Number of fixed queries:", len(fixed_queries))

Number of chunks: 74
Number of fixed queries: 8


In [15]:
corpus_texts = [chunk["text"] for chunk in corpus]

corpus_embeddings = embedding_model.encode_document(
    corpus_texts,
    convert_to_tensor=True,
    show_progress_bar=True
)

print("Corpus embeddings shape:", corpus_embeddings.shape)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Corpus embeddings shape: torch.Size([74, 768])


In [16]:
def embedding_top_k(query, k=3):
    query_embedding = embedding_model.encode_query(
        query,
        convert_to_tensor=True
    )

    similarities = embedding_model.similarity(
        query_embedding,
        corpus_embeddings
    )[0]

    ranked_indices = similarities.argsort(descending=True)[:k].tolist()

    results = []

    for rank, i in enumerate(ranked_indices, start=1):
        results.append({
            "rank": rank,
            "id": corpus[i]["id"],
            "title": corpus[i]["title"],
            "first_line": first_line(corpus[i]["text"]),
            "score": float(similarities[i])
        })

    return results

In [17]:
test_query = fixed_queries[0]

print("Query:", test_query["id"])
print(test_query["text"])
print("Relevant:", test_query["relevant"])

results = embedding_top_k(test_query["text"], k=3)

for result in results:
    print(
        f'{result["rank"]}. '
        f'{result["id"]} | '
        f'{result["first_line"]} | '
        f'score = {result["score"]:.4f}'
    )

Query: Q-01
What is form GR-305 used for?
Relevant: ['GO-08', 'GO-12', 'GO-23']
1. GO-08 | Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision they believe is wrong and which rule in this handbook supports their case. | score = 0.6955
2. GO-12 | A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using form GR-305. The appeal is read by a panel of three staff members who did not take part in the original decision, and the panel answers within 20 working days. An appeal reviews the decision on the evidence that was on file; new income information cannot be added through an appeal and needs a new application. | score = 0.4609
3. GO-23 | Если студент не согласен с решением, он может подать апелляцию по форме GR-305 в течение 14 календарных дн

In [18]:
embedding_results = {}

for query in fixed_queries:
    query_id = query["id"]
    query_text = query["text"]
    relevant = query["relevant"]

    results = embedding_top_k(query_text, k=3)
    embedding_results[query_id] = results

    retrieved_ids = [
        result["id"]
        for result in results
    ]

    hit_at_3 = any(
        chunk_id in relevant
        for chunk_id in retrieved_ids
    )

    print("=" * 100)
    print(f"{query_id}: {query_text}")
    print(f"Relevant chunks: {relevant}")
    print()

    for result in results:
        print(
            f'{result["rank"]}. '
            f'{result["id"]} | '
            f'{result["first_line"]} | '
            f'score = {result["score"]:.4f}'
        )

    print()
    print("hit@3:", hit_at_3)
    print()

Q-01: What is form GR-305 used for?
Relevant chunks: ['GO-08', 'GO-12', 'GO-23']

1. GO-08 | Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision they believe is wrong and which rule in this handbook supports their case. | score = 0.6955
2. GO-12 | A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using form GR-305. The appeal is read by a panel of three staff members who did not take part in the original decision, and the panel answers within 20 working days. An appeal reviews the decision on the evidence that was on file; new income information cannot be added through an appeal and needs a new application. | score = 0.4609
3. GO-23 | Если студент не согласен с решением, он может подать апелляцию по форме GR-305 в течение 14 календарных 

In [19]:
embedding_hits = {}

for query in fixed_queries:
    query_id = query["id"]
    relevant = query["relevant"]

    retrieved_ids = [
        result["id"]
        for result in embedding_results[query_id]
    ]

    embedding_hits[query_id] = any(
        chunk_id in relevant
        for chunk_id in retrieved_ids
    )


print("EmbeddingGemma hit@3 summary")
print("-" * 40)

for query_id, hit in embedding_hits.items():
    print(f"{query_id}: {hit}")

total_embedding_hits = sum(embedding_hits.values())

print()
print(f"Total: {total_embedding_hits} / {len(fixed_queries)}")

EmbeddingGemma hit@3 summary
----------------------------------------
Q-01: True
Q-02: True
Q-03: True
Q-04: True
Q-05: True
Q-06: True
Q-07: True
Q-08: True

Total: 8 / 8


### My own queries

I add two queries before looking at their retrieval results.

**Q-09 — expected BM25 win**

`Where is room 214 of the Grant Office?`

Relevant chunk: `GO-21`

I expect BM25 to perform well because the exact rare string `room 214`
appears in the relevant chunk.

**Q-10 — expected EmbeddingGemma win**

`Can someone else receive my grant payment for me?`

Relevant chunk: `GO-15`

I expect EmbeddingGemma to perform better because the relevant chunk is in
Russian and expresses the same idea semantically: the grant cannot be transferred
to a parent's or another person's account. There are few exact English word matches.

In [20]:
own_queries = [
    {
        "id": "Q-09",
        "text": "Where is room 214 of the Grant Office?",
        "relevant": ["GO-21"]
    },
    {
        "id": "Q-10",
        "text": "Can someone else receive my grant payment for me?",
        "relevant": ["GO-15"]
    }
]

all_queries = fixed_queries + own_queries

print("Total queries:", len(all_queries))

for query in own_queries:
    print(
        query["id"],
        "|",
        query["text"],
        "| relevant:",
        query["relevant"]
    )

Total queries: 10
Q-09 | Where is room 214 of the Grant Office? | relevant: ['GO-21']
Q-10 | Can someone else receive my grant payment for me? | relevant: ['GO-15']


In [21]:
for query in own_queries:
    query_id = query["id"]
    query_text = query["text"]
    relevant = query["relevant"]

    print("=" * 100)
    print(f"{query_id}: {query_text}")
    print(f"Relevant chunks: {relevant}")

    # BM25
    bm25_res = bm25_top_k(query_text, k=3)
    bm25_ids = [result["id"] for result in bm25_res]
    bm25_hit = any(chunk_id in relevant for chunk_id in bm25_ids)

    print("\nBM25:")
    for result in bm25_res:
        print(
            f'{result["rank"]}. '
            f'{result["id"]} | '
            f'{result["first_line"]} | '
            f'score = {result["score"]:.4f}'
        )

    print("BM25 hit@3:", bm25_hit)

    # EmbeddingGemma
    emb_res = embedding_top_k(query_text, k=3)
    emb_ids = [result["id"] for result in emb_res]
    emb_hit = any(chunk_id in relevant for chunk_id in emb_ids)

    print("\nEmbeddingGemma:")
    for result in emb_res:
        print(
            f'{result["rank"]}. '
            f'{result["id"]} | '
            f'{result["first_line"]} | '
            f'score = {result["score"]:.4f}'
        )

    print("Embedding hit@3:", emb_hit)
    print()

Q-09: Where is room 214 of the Grant Office?
Relevant chunks: ['GO-21']

BM25:
1. GO-21 | The Grant Office is in room 214 of the main building and is open Monday to Friday from 10:00 to 17:00, with a lunch break from 13:00 to 14:00. Questions can also be sent through the 'Grants' section of the student portal, which is answered within three working days. The front desk can explain the rules but cannot change a decision or accept documents on paper. | score = 17.1277
2. GO-22 | The documents an applicant uploads are used only to decide the grant. They are kept for three years after the last payment and then deleted. The Grant Office does not share income information with lecturers, with other students, or with any organisation outside the university, except where the law requires it. | score = 10.5756
3. GO-02 | An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their household is 

In [22]:
comparison_rows = []

for query in all_queries:
    query_id = query["id"]
    query_text = query["text"]
    relevant = query["relevant"]

    bm25_res = bm25_top_k(query_text, k=3)
    emb_res = embedding_top_k(query_text, k=3)

    bm25_ids = [r["id"] for r in bm25_res]
    emb_ids = [r["id"] for r in emb_res]

    bm25_hit = any(chunk_id in relevant for chunk_id in bm25_ids)
    emb_hit = any(chunk_id in relevant for chunk_id in emb_ids)

    comparison_rows.append({
        "Query": query_id,
        "BM25 top-3": ", ".join(bm25_ids),
        "BM25 hit@3": bm25_hit,
        "Embedding top-3": ", ".join(emb_ids),
        "Embedding hit@3": emb_hit
    })

In [23]:
import pandas as pd

comparison_df = pd.DataFrame(comparison_rows)

comparison_df

,Query,BM25 top-3,BM25 hit@3,Embedding top-3,Embedding hit@3
0,Q-01,"GO-08, GO-12, GO-06",True,"GO-08, GO-12, GO-23",True
1,Q-02,"KZ-17, GO-20, KZ-25",False,"GO-01, GO-10, GO-20",True
2,Q-03,"GO-14, KZ-01, KZ-02",True,"GO-14, GO-09, GO-11",True
3,Q-04,"GO-05, GO-17, GO-19",False,"GO-02, GO-05, GO-17",True
4,Q-05,"KZ-05, KZ-06, KZ-11",True,"KZ-05, KZ-06, KZ-03",True
5,Q-06,"KZ-37, KZ-38, KZ-03",True,"KZ-37, KZ-38, KZ-10",True
6,Q-07,"GO-21, GO-16, GO-06",False,"GO-15, GO-18, GO-01",True
7,Q-08,"GO-17, GO-05, GO-04",True,"GO-17, GO-18, GO-15",True
8,Q-09,"GO-21, GO-22, GO-02",True,"GO-21, GO-19, GO-06",True
9,Q-10,"GO-22, GO-08, GO-13",False,"GO-18, GO-15, GO-22",True


In [24]:
bm25_total = comparison_df["BM25 hit@3"].sum()
embedding_total = comparison_df["Embedding hit@3"].sum()

print(f"BM25 hit@3: {bm25_total}/{len(comparison_df)}")
print(f"EmbeddingGemma hit@3: {embedding_total}/{len(comparison_df)}")

BM25 hit@3: 6/10
EmbeddingGemma hit@3: 10/10


In [25]:
q11 = {
    "id": "Q-11",
    "text": "What is form GR-104 used for?",
    "relevant": ["GO-06"]
}

print("Q-11:", q11["text"])
print("Relevant:", q11["relevant"])

bm25_q11 = bm25_top_k(q11["text"], k=3)
emb_q11 = embedding_top_k(q11["text"], k=3)

print("\nBM25:")
for r in bm25_q11:
    print(r["rank"], r["id"], f'{r["score"]:.4f}')

print("\nEmbeddingGemma:")
for r in emb_q11:
    print(r["rank"], r["id"], f'{r["score"]:.4f}')

Q-11: What is form GR-104 used for?
Relevant: ['GO-06']

BM25:
1 GO-06 9.7966
2 GO-08 8.4484
3 KZ-43 4.4941

EmbeddingGemma:
1 GO-06 0.6270
2 GO-08 0.4976
3 GO-07 0.4455


In [26]:
q12 = {
    "id": "Q-12",
    "text": "GR-210",
    "relevant": ["GO-10", "GO-13"]
}

print("Q-12:", q12["text"])
print("Relevant:", q12["relevant"])

bm25_q12 = bm25_top_k(q12["text"], k=3)
emb_q12 = embedding_top_k(q12["text"], k=3)

print("\nBM25:")
for r in bm25_q12:
    print(r["rank"], r["id"], f'{r["score"]:.4f}')

print("\nEmbeddingGemma:")
for r in emb_q12:
    print(r["rank"], r["id"], f'{r["score"]:.4f}')

Q-12: GR-210
Relevant: ['GO-10', 'GO-13']

BM25:
1 GO-07 4.3456
2 GO-13 3.1760
3 GO-10 3.1062

EmbeddingGemma:
1 GO-07 0.5736
2 GO-10 0.4288
3 GO-13 0.4187


### Additional lexical test

**Q-13 — expected BM25 win**

`What rule mentions 400,000 tenge?`

Relevant chunk: `GO-16`

I expect BM25 to perform better because the exact numeric string
`400,000 tenge` appears in the relevant chunk. Numeric values and exact
strings are a case where lexical matching can be more reliable than semantic similarity.

In [27]:
q13 = {
    "id": "Q-13",
    "text": "What rule mentions 400,000 tenge?",
    "relevant": ["GO-16"]
}

print("Q-13:", q13["text"])
print("Relevant:", q13["relevant"])

bm25_q13 = bm25_top_k(q13["text"], k=3)
emb_q13 = embedding_top_k(q13["text"], k=3)

print("\nBM25:")
for r in bm25_q13:
    print(r["rank"], r["id"], f'{r["score"]:.4f}')

print("\nEmbeddingGemma:")
for r in emb_q13:
    print(r["rank"], r["id"], f'{r["score"]:.4f}')

Q-13: What rule mentions 400,000 tenge?
Relevant: ['GO-16']

BM25:
1 GO-16 8.6247
2 GO-03 7.6887
3 GO-05 6.7566

EmbeddingGemma:
1 GO-16 0.4308
2 GO-05 0.4005
3 GO-03 0.3626


### B3 — Comparison of the two retrievers

Across the ten main queries, BM25 achieved hit@3 on 6/10 queries,
while EmbeddingGemma achieved hit@3 on 10/10.

#### 1. Query where both retrievers agreed

For Q-06, `Where did the Soviet Union test its nuclear weapons?`,
both retrievers returned the same two relevant chunks first:

- BM25: KZ-37, KZ-38, KZ-03
- EmbeddingGemma: KZ-37, KZ-38, KZ-10

This query was relatively easy because the wording of the query is very close
to the corpus. KZ-37 explicitly contains `Soviet Union's nuclear weapons`,
and KZ-38 describes Soviet nuclear tests at Semipalatinsk.

Both lexical matching and semantic similarity therefore point to the same chunks.

#### 2. BM25 lexical-strength example

In my ten main queries I did not get a strict case where BM25 ranked the relevant
chunk higher than EmbeddingGemma.

The closest example is Q-09, `Where is room 214 of the Grant Office?`.

Both retrievers ranked GO-21 first, but BM25 had a very strong exact lexical
match on the rare string `room 214`.

This shows the type of query where BM25 is strong: exact identifiers, numbers,
codes, or uncommon strings. However, in this run EmbeddingGemma also retrieved
GO-21 at rank 1, so this was a tie rather than a strict BM25 win.

I also tested additional exact-match queries such as `GR-210` and
`400,000 tenge`, but EmbeddingGemma still ranked the relevant chunks as well
as BM25. I therefore did not claim a BM25 win that was not present in my outputs.

#### 3. Query where EmbeddingGemma did better

Q-07, `Can the grant be sent to my mother's card?`, is a clear case where
EmbeddingGemma performed better.

BM25 returned:

1. GO-21
2. GO-16
3. GO-06

and missed the relevant chunk GO-15 completely.

EmbeddingGemma returned:

1. GO-15
2. GO-18
3. GO-01

GO-15 is written in Russian and says that the bank account must belong to the
student and that payment to a parent's or another person's account is impossible.

The English query uses `mother's card`, while the relevant chunk expresses the
same meaning in Russian. BM25 cannot match these words across languages, while
EmbeddingGemma can represent their semantic similarity.

This is why EmbeddingGemma retrieved the correct chunk and BM25 did not.

---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

## Part C — Connect retrieval to the model

The pipeline is:

question → retrieve top-3 chunks → build prompt → gpt-5.6-luna → answer

For the RAG systems I use the same BM25 and EmbeddingGemma retrievers
from Part B and retrieve the top 3 chunks unless stated otherwise.

In [28]:
!ls
!ls data

data  HW3.ipynb  README.md  requirements.txt
corpus.jsonl  qa.json  queries.json  SOURCES.md


In [29]:
import json

with open("data/qa.json", "r", encoding="utf-8") as f:
    qa_data = json.load(f)

print(type(qa_data))
print(json.dumps(qa_data, indent=2, ensure_ascii=False))

<class 'dict'>
{
  "note": "Part C's six questions, each asked of all three systems. `reference` is what the corpus says; `relevant` lists the chunks that hold it. C-06 has no answer in the corpus on purpose: the right answer from a RAG system is that it does not know.",
  "questions": [
    {
      "id": "C-01",
      "question": "How much does a student in income band 2 receive, and when is it paid?",
      "reference": "150,000 tenge for the academic year, in two equal instalments: October and February.",
      "relevant": [
        "GO-04",
        "GO-15"
      ]
    },
    {
      "id": "C-02",
      "question": "My grant application was refused. How long do I have to appeal, and which form do I use?",
      "reference": "14 calendar days from the date on the decision letter, using form GR-305.",
      "relevant": [
        "GO-08",
        "GO-12",
        "GO-23"
      ]
    },
    {
      "id": "C-03",
      "question": "What is the minimum GPA needed to qualify for the need-b

## C1 — the complete prompt, printed before it is sent

In [30]:
import json

corpus = []

with open("data/corpus.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        corpus.append(json.loads(line))

with open("data/qa.json", "r", encoding="utf-8") as f:
    qa_data = json.load(f)

questions = qa_data["questions"]

print("Chunks:", len(corpus))
print("Questions:", len(questions))

Chunks: 74
Questions: 6


In [31]:
def embedding_top_k(query, k=3):
    query_embedding = embedding_model.encode_query(
        query,
        convert_to_tensor=True
    )

    similarities = embedding_model.similarity(
        query_embedding,
        corpus_embeddings
    )[0]

    ranked_indices = similarities.argsort(
        descending=True
    )[:k].tolist()

    results = []

    for rank, i in enumerate(ranked_indices, start=1):
        results.append({
            "rank": rank,
            "index": i,
            "id": corpus[i]["id"],
            "title": corpus[i]["title"],
            "text": corpus[i]["text"],   # <-- этого не хватало
            "score": float(similarities[i])
        })

    return results

In [35]:
c1_results = embedding_top_k(
    c1_question["question"],
    k=3
)

In [34]:
print(c1_results[0].keys())

dict_keys(['rank', 'index', 'id', 'title', 'text', 'score'])


In [33]:
c1_question = next(
    q for q in questions
    if q["id"] == "C-03"
)

print("Question ID:", c1_question["id"])
print("Question:", c1_question["question"])
print("Reference:", c1_question["reference"])
print("Relevant:", c1_question["relevant"])

c1_results = embedding_top_k(
    c1_question["question"],
    k=3
)

print("\nEmbeddingGemma top-3:")

for result in c1_results:
    print(
        result["rank"],
        result["id"],
        f'score = {result["score"]:.4f}',
        "|",
        result["title"]
    )

Question ID: C-03
Question: What is the minimum GPA needed to qualify for the need-based grant?
Reference: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)
Relevant: ['GO-02']

EmbeddingGemma top-3:
1 GO-02 score = 0.6286 | Who qualifies
2 GO-20 score = 0.5691 | Who cannot apply
3 GO-05 score = 0.5451 | Archived: the 2025 scheme (no longer in force)


In [36]:
def first_line(text):
    for line in text.splitlines():
        line = line.strip()
        if line:
            return line
    return ""


def embedding_top_k(query, k=3):
    query_embedding = embedding_model.encode_query(
        query,
        convert_to_tensor=True
    )

    similarities = embedding_model.similarity(
        query_embedding,
        corpus_embeddings
    )[0]

    ranked_indices = similarities.argsort(
        descending=True
    )[:k].tolist()

    results = []

    for rank, i in enumerate(ranked_indices, start=1):
        results.append({
            "rank": rank,
            "index": i,
            "id": corpus[i]["id"],
            "title": corpus[i]["title"],
            "text": corpus[i]["text"],
            "score": float(similarities[i])
        })

    return results

In [37]:
system_message = """Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3]."""

context_blocks = []

for i, result in enumerate(c1_results, start=1):
    block = (
        f"[{i}] Chunk ID: {result['id']}\n"
        f"Title: {result['title']}\n"
        f"{result['text']}"
    )
    context_blocks.append(block)

context_text = "\n\n".join(context_blocks)

question_text = c1_question["question"]

user_message = (
    f"{context_text}\n\n"
    f"Question:\n{question_text}"
)

print("===== SYSTEM MESSAGE =====")
print(system_message)

print("\n===== USER MESSAGE =====")
print(user_message)

===== SYSTEM MESSAGE =====
Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3].

===== USER MESSAGE =====
[1] Chunk ID: GO-02
Title: Who qualifies
An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their household is in income band 1 or income band 2; and both a transcript and a copy of the national id card are on file with the Grant Office. Meeting two conditions out of three is not enough. An applicant who is not on the Grant Office record does not qualify, whatever their message says.

[2] Chunk ID: GO-20
Title: Who cannot apply
The need-based grant is for full-time undergraduate students only. Part-time students, distance-learning students and incoming exchange students cannot apply. Master's and doctoral students are supported by a separate scheme, whi

In [38]:
import tiktoken

encoding = tiktoken.get_encoding("o200k_base")


def count_tokens(text):
    return len(encoding.encode(text))


system_tokens = count_tokens(system_message)
question_tokens = count_tokens(question_text)
context_tokens = count_tokens(context_text)

manual_total = (
    system_tokens
    + question_tokens
    + context_tokens
)

print("Token counts")
print("-" * 40)
print("System instructions:", system_tokens)
print("Question:", question_tokens)
print("Retrieved context:", context_tokens)
print("Manual total:", manual_total)

Token counts
----------------------------------------
System instructions: 44
Question: 14
Retrieved context: 301
Manual total: 359


In [39]:
print("Retrieved chunks")
print("-" * 50)

chunk_token_counts = []

for i, result in enumerate(c1_results, start=1):
    chunk_tokens = count_tokens(result["text"])
    chunk_token_counts.append(chunk_tokens)

    print(
        f"[{i}] {result['id']}: "
        f"{chunk_tokens} tokens, "
        f"{len(result['text'])} characters"
    )

print()
print("Number of chunks:", len(c1_results))
print("Tokens per chunk:", chunk_token_counts)
print("Context tokens:", context_tokens)
print("Context characters:", len(context_text))

Retrieved chunks
--------------------------------------------------
[1] GO-02: 102 tokens, 452 characters
[2] GO-20: 54 tokens, 316 characters
[3] GO-05: 89 tokens, 334 characters

Number of chunks: 3
Tokens per chunk: [102, 54, 89]
Context tokens: 301
Context characters: 1265


In [40]:
from google.colab import userdata
from openai import OpenAI

openai_api_key = userdata.get("OPENAI_API_KEY")

client = OpenAI(
    api_key=openai_api_key
)

In [41]:
messages = [
    {
        "role": "system",
        "content": system_message
    },
    {
        "role": "user",
        "content": user_message
    }
]

response = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=messages
)

answer = response.choices[0].message.content

print("===== MODEL ANSWER =====")
print(answer)

print("\n===== API USAGE =====")
print("prompt_tokens:", response.usage.prompt_tokens)
print("completion_tokens:", response.usage.completion_tokens)
print("total_tokens:", response.usage.total_tokens)

===== MODEL ANSWER =====
The minimum GPA needed is **2.67 on a 4.0 scale**. [1]

===== API USAGE =====
prompt_tokens: 371
completion_tokens: 24
total_tokens: 395


In [42]:
api_prompt_tokens = response.usage.prompt_tokens

difference = api_prompt_tokens - manual_total

print("Manual content-token count:", manual_total)
print("API prompt_tokens:", api_prompt_tokens)
print("Difference:", difference)

Manual content-token count: 359
API prompt_tokens: 371
Difference: 12


### C1 — Token accounting

| Part of the input | Tokens (`tiktoken` `o200k_base`) |
|---|---:|
| System instructions | 44 |
| Question | 14 |
| Retrieved context | 301 |
| **Total input** | **359** |
| `usage.prompt_tokens` from the response | **371** |

- Number of retrieved chunks: **3**
- Tokens per chunk: **102, 54, 89**
- Total context size: **301 tokens / 1265 characters**

### Why do the `tiktoken` total and `usage.prompt_tokens` differ?

The manually counted input contains 359 tokens, while the API reports 371 prompt tokens, so the difference is 12 tokens.

The manual `tiktoken` count includes only the visible text content. The API also encodes the chat message structure, including the system and user roles and the formatting used internally around the messages. Because of this, `usage.prompt_tokens` is slightly higher than the manual count.

### What exactly does retrieval add to the model's input?

Retrieval adds the text of the selected corpus chunks to the prompt.

For C-03, EmbeddingGemma retrieved:

1. `GO-02` — the current 2026 qualification rule
2. `GO-20` — eligibility restrictions
3. `GO-05` — the archived 2025 rule

These three chunks were inserted as numbered context blocks before the question.

The model does not search or open the original documents itself. It only receives the retrieved text that the code adds to the prompt.

### Model answer

`The minimum GPA is 2.67 on the 4.0 scale. [1]`

This answer agrees with the reference and correctly uses the current rule in `GO-02` instead of the archived 3.0 rule in `GO-05`.

## C2 — three systems, six questions

In [43]:
import re
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(
        r"\b[\w]+(?:-[\w]+)*\b",
        text.lower()
    )

tokenized_corpus = [
    tokenize(chunk["text"])
    for chunk in corpus
]

bm25 = BM25Okapi(
    tokenized_corpus,
    k1=1.5,
    b=0.75
)

def bm25_top_k_full(query, k=3):
    scores = bm25.get_scores(tokenize(query))

    ranked_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )[:k]

    results = []

    for rank, i in enumerate(ranked_indices, start=1):
        results.append({
            "rank": rank,
            "index": i,
            "id": corpus[i]["id"],
            "title": corpus[i]["title"],
            "text": corpus[i]["text"],
            "score": float(scores[i])
        })

    return results

print("BM25 ready.")

BM25 ready.


In [44]:
system_message = """Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3]."""

def build_rag_user_message(question, retrieval_results):
    blocks = []

    for i, result in enumerate(retrieval_results, start=1):
        block = (
            f"[{i}] Chunk ID: {result['id']}\n"
            f"Title: {result['title']}\n"
            f"{result['text']}"
        )
        blocks.append(block)

    context = "\n\n".join(blocks)

    user_message = (
        f"{context}\n\n"
        f"Question:\n{question}"
    )

    return user_message, context

In [45]:
def ask_llm_only(question):
    response = client.chat.completions.create(
        model="gpt-5.6-luna",
        messages=[
            {
                "role": "user",
                "content": question
            }
        ]
    )

    return response.choices[0].message.content


def ask_rag(question, retrieval_results):
    user_message, context = build_rag_user_message(
        question,
        retrieval_results
    )

    response = client.chat.completions.create(
        model="gpt-5.6-luna",
        messages=[
            {
                "role": "system",
                "content": system_message
            },
            {
                "role": "user",
                "content": user_message
            }
        ]
    )

    return (
        response.choices[0].message.content,
        context
    )

In [46]:
c2_runs = []

for q in questions:
    qid = q["id"]
    question = q["question"]
    reference = q["reference"]

    print("=" * 100)
    print(qid, "-", question)

    # 1. LLM only
    llm_answer = ask_llm_only(question)

    # 2. BM25 RAG
    bm25_results_c2 = bm25_top_k_full(
        question,
        k=3
    )

    bm25_answer, bm25_context = ask_rag(
        question,
        bm25_results_c2
    )

    # 3. Embedding RAG
    embedding_results_c2 = embedding_top_k(
        question,
        k=3
    )

    embedding_answer, embedding_context = ask_rag(
        question,
        embedding_results_c2
    )

    c2_runs.append({
        "id": qid,
        "question": question,
        "reference": reference,

        "llm_answer": llm_answer,

        "bm25_answer": bm25_answer,
        "bm25_results": bm25_results_c2,
        "bm25_context": bm25_context,

        "embedding_answer": embedding_answer,
        "embedding_results": embedding_results_c2,
        "embedding_context": embedding_context
    })

    print("done")

C-01 - How much does a student in income band 2 receive, and when is it paid?
done
C-02 - My grant application was refused. How long do I have to appeal, and which form do I use?
done
C-03 - What is the minimum GPA needed to qualify for the need-based grant?
done
C-04 - When did Kazakhstan decide to switch the Kazakh alphabet from Cyrillic to Latin, and when is the switch now due to finish?
done
C-05 - When was Narxoz University founded, and what was it called at the start?
done
C-06 - I am a master's student. How much need-based grant can I get?
done


In [47]:
for run in c2_runs:
    print("=" * 100)
    print(run["id"])
    print("QUESTION:")
    print(run["question"])

    print("\nREFERENCE:")
    print(run["reference"])

    print("\nLLM ONLY:")
    print(run["llm_answer"])

    print("\nBM25 RAG:")
    print(run["bm25_answer"])

    print("\nEMBEDDING RAG:")
    print(run["embedding_answer"])

    print()

C-01
QUESTION:
How much does a student in income band 2 receive, and when is it paid?

REFERENCE:
150,000 tenge for the academic year, in two equal instalments: October and February.

LLM ONLY:
If you mean **Northern Ireland’s Education Maintenance Allowance (EMA)**, a student in **Income Band 2 receives £20 per week**. It is normally **paid fortnightly in arrears**, subject to meeting the attendance and learning-agreement requirements.

BM25 RAG:
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments**: the first in **October** and the second in **February**. [1]

EMBEDDING RAG:
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments**: **75,000 tenge in October** and **75,000 tenge in February**. [1]

C-02
QUESTION:
My grant application was refused. How long do I have to appeal, and which form do I use?

REFERENCE:
14 calendar days from the date on the decision letter, using for

### C2 — Three systems on six questions

| Question | LLM only | BM25 RAG | Embedding RAG |
|---|---|---|---|
| C-01 | ✗ wrong scheme: £20/week | ✓ 150,000 tenge; Oct + Feb | ✓ 150,000 tenge; Oct + Feb |
| C-02 | ✗ no exact deadline/form | ✓ 14 days; GR-305 | ✓ 14 days; GR-305 |
| C-03 | ✗ guessed 2.0–2.5 | ✓ GPA 2.67 | ✓ GPA 2.67 |
| C-04 | ✓ Oct 2017; finish 2031 | ✓ Oct 2017; finish 2031 | ✓ Oct 2017; finish 2031 |
| C-05 | ✓ 1963; Institute of National Economy | ✓ 1963; Almaty Institute of National Economy | ✓ 1963; Almaty Institute of National Economy |
| C-06 | ✗ invented external grant information | ✓ separate scheme; amount not specified | ✓ separate scheme; amount unknown |
| **agrees with reference** | **2/6** | **6/6** | **6/6** |

### C2 observations

The LLM-only system agreed with the reference on 2 of the 6 questions.

It answered the general Kazakhstan questions C-04 and C-05 correctly, which is
consistent with the fact that this information may already be known by the model.

However, it failed on the fictional Grant Office questions because those rules
are not available without retrieval. For example, on C-01 it answered about a
different grant scheme and currency, and on C-03 it guessed a GPA range instead
of the required 2.67.

Both RAG systems agreed with the reference on all 6 questions.

The retrieved Grant Office chunks gave the model the exact fictional rules,
while retrieval also grounded the Kazakhstan answers in the corpus.

## C3 — question 1 of 3: (id)

**1. What BM25 retrieved** (top 3, ids and scores):

>

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

>

**3. What was sent to the model** (the context blocks):

>

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

>

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

>

In [48]:
selected_c3_ids = ["C-01", "C-03", "C-06"]

c3_runs = [
    run for run in c2_runs
    if run["id"] in selected_c3_ids
]

print("Selected questions:")
for run in c3_runs:
    print(run["id"], "-", run["question"])

Selected questions:
C-01 - How much does a student in income band 2 receive, and when is it paid?
C-03 - What is the minimum GPA needed to qualify for the need-based grant?
C-06 - I am a master's student. How much need-based grant can I get?


In [49]:
for run in c3_runs:
    print("=" * 100)
    print(run["id"], "-", run["question"])

    print("\nBM25 top-3:")
    for r in run["bm25_results"]:
        print(
            f'{r["rank"]}. {r["id"]} | '
            f'score = {r["score"]:.4f} | '
            f'{r["title"]}'
        )

    print("\nEmbeddingGemma top-3:")
    for r in run["embedding_results"]:
        print(
            f'{r["rank"]}. {r["id"]} | '
            f'score = {r["score"]:.4f} | '
            f'{r["title"]}'
        )

    print()

C-01 - How much does a student in income band 2 receive, and when is it paid?

BM25 top-3:
1. GO-04 | score = 20.2924 | How much the grant pays
2. GO-02 | score = 18.8491 | Who qualifies
3. GO-01 | score = 15.2915 | About the need-based study grant 2026

EmbeddingGemma top-3:
1. GO-04 | score = 0.5845 | How much the grant pays
2. GO-03 | score = 0.4842 | Income bands
3. GO-15 | score = 0.4506 | Выплата гранта

C-03 - What is the minimum GPA needed to qualify for the need-based grant?

BM25 top-3:
1. GO-02 | score = 14.0110 | Who qualifies
2. GO-05 | score = 13.7794 | Archived: the 2025 scheme (no longer in force)
3. GO-11 | score = 12.4064 | When a document is missing

EmbeddingGemma top-3:
1. GO-02 | score = 0.6286 | Who qualifies
2. GO-20 | score = 0.5691 | Who cannot apply
3. GO-05 | score = 0.5451 | Archived: the 2025 scheme (no longer in force)

C-06 - I am a master's student. How much need-based grant can I get?

BM25 top-3:
1. GO-16 | score = 13.1204 | Combining the grant with o

In [50]:
for run in c3_runs:
    print("=" * 100)
    print(run["id"])

    print("\n===== BM25 CONTEXT SENT TO MODEL =====")
    print(run["bm25_context"])

    print("\n===== EMBEDDING CONTEXT SENT TO MODEL =====")
    print(run["embedding_context"])

    print()

C-01

===== BM25 CONTEXT SENT TO MODEL =====
[1] Chunk ID: GO-04
Title: How much the grant pays
The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.

[2] Chunk ID: GO-02
Title: Who qualifies
An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their household is in income band 1 or income band 2; and both a transcript and a copy of the national id card are on file with the Grant Office. Meeting two conditions out of three is not enough. An applicant who is not on the Grant Office record does not qualify, whatever their message says.

[3] Chunk ID: GO-01


In [51]:
for run in c3_runs:
    print("=" * 100)
    print(run["id"], "-", run["question"])

    print("\nREFERENCE:")
    print(run["reference"])

    print("\nBM25 RAG ANSWER:")
    print(run["bm25_answer"])

    print("\nEMBEDDING RAG ANSWER:")
    print(run["embedding_answer"])

    print()

C-01 - How much does a student in income band 2 receive, and when is it paid?

REFERENCE:
150,000 tenge for the academic year, in two equal instalments: October and February.

BM25 RAG ANSWER:
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments**: the first in **October** and the second in **February**. [1]

EMBEDDING RAG ANSWER:
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments**: **75,000 tenge in October** and **75,000 tenge in February**. [1]

C-03 - What is the minimum GPA needed to qualify for the need-based grant?

REFERENCE:
A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)

BM25 RAG ANSWER:
The minimum GPA is **2.67 on a 4.0 scale**. [1]

EMBEDDING RAG ANSWER:
The minimum GPA is **2.67 on a 4.0 scale**. [1]

C-06 - I am a master's student. How much need-based grant can I get?

REFERENCE:
The corpus does not say. GO-

## C3 — Three questions taken apart

### C-01 — Grant amount and payment dates

**Question:**  
`How much does a student in income band 2 receive, and when is it paid?`

**Reference:**  
150,000 tenge for the academic year, in two equal instalments: October and February.

#### BM25 retrieval

| Rank | Chunk | Score | Title |
|---|---|---:|---|
| 1 | GO-04 | 20.2924 | How much the grant pays |
| 2 | GO-02 | 18.8491 | Who qualifies |
| 3 | GO-01 | 15.2915 | About the need-based study grant 2026 |

The answer was already fully present in BM25's first chunk, GO-04.

#### EmbeddingGemma retrieval

| Rank | Chunk | Score | Title |
|---|---|---:|---|
| 1 | GO-04 | 0.5845 | How much the grant pays |
| 2 | GO-03 | 0.4842 | Income bands |
| 3 | GO-15 | 0.4506 | Выплата гранта |

EmbeddingGemma also placed GO-04 first. It additionally retrieved GO-15, a Russian
chunk confirming that the payments are made in October and February.

#### Answers

**BM25 RAG:**  
`A student in income band 2 receives 150,000 tenge for the academic year, paid in two equal instalments: the first in October and the second in February. [1]`

**Embedding RAG:**  
`A student in income band 2 receives 150,000 tenge for the academic year, paid in two equal instalments: 75,000 tenge in October and 75,000 tenge in February. [1]`

#### Analysis

Retrieval made the answer more accurate and grounded. The LLM-only system had
answered using an unrelated grant scheme in pounds, while both RAG systems received
GO-04, which contains the exact amount and payment schedule.

There was no retrieval failure: the required fact was present in the first retrieved
chunk for both systems.

There was also no generation failure: both answers correctly used the retrieved fact.
The Embedding RAG answer additionally inferred that each of two equal instalments is
75,000 tenge, which is consistent with the context.

**Failure type:** none.

### C-03 — Current GPA rule versus archived rule

**Question:**  
`What is the minimum GPA needed to qualify for the need-based grant?`

**Reference:**  
A cumulative GPA of at least 2.67 on the 4.0 scale. The 3.0 value in GO-05 is the
archived 2025 rule.

#### BM25 retrieval

| Rank | Chunk | Score | Title |
|---|---|---:|---|
| 1 | GO-02 | 14.0110 | Who qualifies |
| 2 | GO-05 | 13.7794 | Archived: the 2025 scheme (no longer in force) |
| 3 | GO-11 | 12.4064 | When a document is missing |

BM25 retrieved the correct current rule first, but it also ranked the archived
2025 rule very highly at rank 2.

GO-02 says that the current minimum GPA is 2.67. GO-05 contains the exact phrase
about a minimum GPA but gives the old value of 3.0 and explicitly says that the rule
no longer applies.

#### EmbeddingGemma retrieval

| Rank | Chunk | Score | Title |
|---|---|---:|---|
| 1 | GO-02 | 0.6286 | Who qualifies |
| 2 | GO-20 | 0.5691 | Who cannot apply |
| 3 | GO-05 | 0.5451 | Archived: the 2025 scheme (no longer in force) |

EmbeddingGemma also retrieved the correct GO-02 first. The archived GO-05 was still
present, but only at rank 3.

#### Answers

**BM25 RAG:**  
`The minimum GPA is 2.67 on a 4.0 scale. [1]`

**Embedding RAG:**  
`The minimum GPA is 2.67 on a 4.0 scale. [1]`

#### Analysis

This was a riskier retrieval case than C-01 because the context contained two
different GPA values: the current 2.67 rule and the archived 3.0 rule.

BM25 was especially vulnerable to the archived chunk because GO-05 contains the
exact words `minimum GPA`, which closely match the query. However, the correct GO-02
still ranked first.

Both models correctly used GO-02 and ignored the obsolete value in GO-05.

Therefore there was no generation failure. The correct fact was available in the
context and the model selected it correctly.

There was also no retrieval failure because the required chunk GO-02 was retrieved
at rank 1 by both systems. However, BM25's high ranking of the archived rule shows
how exact lexical matching can introduce misleading context even when hit@3 is
technically successful.

**Failure type:** none, but BM25 retrieval introduced a potentially misleading
archived chunk.

### C3 conclusion

None of the six RAG answers in C2 was incorrect, so these three examples do not
contain a complete retrieval or generation failure.

However, they show different levels of retrieval quality:

- C-01 was straightforward: both systems retrieved the exact answer first.
- C-03 shows that successful retrieval can still include misleading context, because
  the archived 3.0 GPA rule was retrieved together with the current 2.67 rule.
- C-06 shows that retrieval quality also depends on ranking: EmbeddingGemma placed
  the decisive "no answer in this handbook" chunk first, while BM25 placed it third.

The distinction is still clear: if the necessary fact had not appeared in the
retrieved context, that would be a retrieval failure. If the fact was present in
the context but the model ignored or contradicted it, that would be a generation
failure. In these three cases, neither failure occurred.

---

In [52]:
topk_question = next(
    q for q in questions
    if q["id"] == "C-03"
)

topk_results = {}

for k in [1, 3, 8]:
    retrieval = embedding_top_k(
        topk_question["question"],
        k=k
    )

    answer, context = ask_rag(
        topk_question["question"],
        retrieval
    )

    topk_results[k] = {
        "retrieval": retrieval,
        "answer": answer,
        "context": context
    }

    print("=" * 100)
    print(f"K = {k}")

    print("\nRetrieved chunks:")
    for r in retrieval:
        print(
            f'{r["rank"]}. {r["id"]} | '
            f'score = {r["score"]:.4f} | '
            f'{r["title"]}'
        )

    print("\nAnswer:")
    print(answer)
    print()

K = 1

Retrieved chunks:
1. GO-02 | score = 0.6286 | Who qualifies

Answer:
The minimum GPA needed is **2.67 on a 4.0 scale**. [1]

K = 3

Retrieved chunks:
1. GO-02 | score = 0.6286 | Who qualifies
2. GO-20 | score = 0.5691 | Who cannot apply
3. GO-05 | score = 0.5451 | Archived: the 2025 scheme (no longer in force)

Answer:
The minimum GPA is **2.67 on a 4.0 scale**. [1]

K = 8

Retrieved chunks:
1. GO-02 | score = 0.6286 | Who qualifies
2. GO-20 | score = 0.5691 | Who cannot apply
3. GO-05 | score = 0.5451 | Archived: the 2025 scheme (no longer in force)
4. GO-16 | score = 0.5378 | Combining the grant with other support
5. GO-01 | score = 0.5347 | About the need-based study grant 2026
6. GO-06 | score = 0.5330 | Form GR-104: the application form
7. GO-19 | score = 0.5265 | Priority when the budget runs short
8. GO-17 | score = 0.5252 | Academic progress during the year

Answer:
The minimum GPA needed is **2.67 on the 4.0 scale**. [1]



In [53]:
print("Top-K context sizes")
print("-" * 40)

for k in [1, 3, 8]:
    context = topk_results[k]["context"]

    tokens = len(
        encoding.encode(context)
    )

    characters = len(context)

    print(
        f"K={k}: "
        f"{tokens} tokens, "
        f"{characters} characters"
    )

Top-K context sizes
----------------------------------------
K=1: 117 tokens, 493 characters
K=3: 301 tokens, 1265 characters
K=8: 816 tokens, 3546 characters


## Final questions

### 1. What is the main difference between BM25 and embedding retrieval?

BM25 mainly retrieves documents by lexical overlap between the query and the document text. Embedding retrieval represents queries and documents as vectors and can retrieve text with similar meaning even when the exact words are different.

In my runs, this difference was clear on Q-07: BM25 missed the relevant Russian chunk, while EmbeddingGemma retrieved it at rank 1 from an English query.

---

### 2. When can BM25 perform better than embeddings?

BM25 can be especially useful when the query contains exact rare strings such as form codes, room numbers, names or numeric values.

For example, Q-09 contained the exact phrase `room 214`, and BM25 retrieved GO-21 at rank 1 with a strong lexical match. In my actual runs, however, I did not find a strict query where BM25 ranked the relevant result higher than EmbeddingGemma, so I would not claim a BM25 win that was not present in the outputs.

---

### 3. When can embeddings perform better than BM25?

Embeddings can perform better when the query and the relevant document express the same meaning using different wording or different languages.

For Q-07, the English query asked whether the grant could be sent to the student's mother's card, while the relevant GO-15 chunk was written in Russian. BM25 missed it completely, but EmbeddingGemma ranked GO-15 first.

---

### 4. Why is cosine similarity used with embeddings?

Cosine similarity measures how closely two embedding vectors point in the same direction. This makes it useful for comparing semantic similarity while reducing the effect of the absolute vector magnitude.

A cosine value closer to 1 means that the query and document embeddings are more similar in the embedding space.

---

### 5. What information is actually sent to an LLM in a RAG system?

The LLM receives the instructions, the user's question and the text of the chunks selected by the retriever.

For C-03, my code retrieved GO-02, GO-20 and GO-05 and inserted those three chunks as numbered context blocks into the prompt. The whole corpus was not sent to the model.

---

### 6. Does the LLM search the original documents itself?

No. In this RAG pipeline, the retriever searches the corpus first and the code adds the selected chunks to the prompt.

The LLM only sees the text that is included in its input. It does not independently open or search `corpus.jsonl`.

---

### 7. What happens if retrieval returns an irrelevant chunk?

An irrelevant chunk consumes prompt tokens and can distract the model from the useful evidence. A more dangerous case is a chunk that is related to the query but contains outdated or conflicting information.

For C-03, both retrievers returned the archived GO-05 chunk containing the old GPA value of 3.0. The model still answered correctly using GO-02, but the archived chunk created unnecessary conflicting context.

---

### 8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse?

A larger Top-K can improve RAG when the answer is spread across several chunks or when the most relevant chunk is not ranked first. However, it also adds more tokens and can introduce irrelevant, duplicated or conflicting information.

For C-03, I tested K = 1, 3 and 8:

| Top-K | Context size | Result |
|---|---:|---|
| 1 | 117 tokens / 493 characters | correct: GPA 2.67 |
| 3 | 301 tokens / 1265 characters | correct: GPA 2.67 |
| 8 | 816 tokens / 3546 characters | correct: GPA 2.67 |

The answer did not improve when K increased. At K=3 the archived GO-05 rule with GPA 3.0 was already added, and at K=8 five more chunks were included. In this example, increasing K mainly increased input size and the risk of distraction without improving the answer.

---

### 9. Is a vector database required to build a RAG system?

No. My implementation stored all corpus embeddings directly in the `corpus_embeddings` tensor and calculated similarity against them in memory.

A vector database can be useful for large collections and efficient search, but it was not required for this 74-chunk corpus.

---

### 10. Which retrieval approach worked better for this corpus, and what evidence supports that?

EmbeddingGemma performed better on retrieval in my experiment.

Across the ten main Part B queries:

- BM25 hit@3: **6/10**
- EmbeddingGemma hit@3: **10/10**

In Part C, both RAG systems still produced answers agreeing with the reference on all six questions:

- BM25 RAG: **6/6**
- Embedding RAG: **6/6**

This means the difference was clearer at the retrieval level than at the final-answer level. BM25 sometimes retrieved weaker or misleading chunks but still included enough correct evidence in the top 3 for the LLM to answer correctly.

---

## Conclusion

If I continued this system, I would combine lexical and embedding retrieval instead of choosing only one. EmbeddingGemma was more reliable on this corpus, especially for paraphrases and cross-language queries, while BM25 still provides useful exact matching for codes, numbers and rare strings.

I would also add metadata filtering or reranking so that archived chunks such as GO-05 are penalized before they reach the model. Finally, I would choose Top-K dynamically rather than always increasing it, because the C-03 experiment showed that more context can increase token usage and introduce conflicting information without improving the answer.